In [4]:
import numpy as np
import pandas as pd
import joblib
from scipy.signal import welch
from scipy.stats import skew, kurtosis
from scipy.integrate import trapezoid

# Set your Sampling Frequency (FS)
# Based on your CSV timestamps (~7.7ms per sample), FS is roughly 128 Hz.
FS = 128 

# ── 1. Feature Extraction Helper Functions ─────────────────────
def band_power(f, pxx, lo, hi):
    idx = (f >= lo) & (f <= hi)
    return float(trapezoid(pxx[idx], f[idx])) if idx.sum() > 0 else 0.0

def spectral_entropy(pxx):
    p = pxx / (pxx.sum() + 1e-10)
    return float(-np.sum(p * np.log2(p + 1e-10)))

def hjorth_params(x):
    d1 = np.diff(x); d2 = np.diff(d1)
    act  = float(np.var(x))
    mob  = float(np.sqrt(np.var(d1) / (act  + 1e-10)))
    comp = float(np.sqrt(np.var(d2) / (np.var(d1) + 1e-10)) / (mob + 1e-10))
    return act, mob, comp

def extract_features(ep):
    nperseg = min(128, len(ep))
    d1 = np.diff(ep)
    half = len(ep) // 2

    # Time-domain (18 features)
    td = [
        float(np.mean(ep)), float(np.std(ep)), float(np.var(ep)),
        float(skew(ep)), float(kurtosis(ep)), float(np.sqrt(np.mean(ep**2))),
        float(np.max(ep) - np.min(ep)), float(np.mean(np.abs(ep))),
        float(np.sum(np.diff(np.sign(ep)) != 0)),
        float(np.percentile(ep, 10)), float(np.percentile(ep, 25)),
        float(np.percentile(ep, 75)), float(np.percentile(ep, 90)),
        float(np.sum(np.abs(d1))),
        float(np.mean(ep[:half])), float(np.mean(ep[half:])),
        float(np.std(ep[:half])), float(np.std(ep[half:]))
    ]

    # Hjorth parameters (3 features)
    act, mob, comp = hjorth_params(ep)
    hj = [act, mob, comp]

    # Autocorrelation lags (5 features)
    ac = np.correlate(ep, ep, mode='full')
    ac = ac[len(ac)//2:]
    ac = ac / (ac[0] + 1e-10)
    autocorr = [float(ac[lag]) if lag < len(ac) else 0.0 for lag in [1, 2, 5, 10, 20]]

    # Frequency-domain (16 features)
    f, pxx = welch(ep, fs=FS, nperseg=nperseg)
    pxx = np.maximum(pxx, 1e-12)

    d_ = band_power(f, pxx, 2,  4)
    t_ = band_power(f, pxx, 4,  8)
    a_ = band_power(f, pxx, 8,  13)
    b_ = band_power(f, pxx, 13, 30)
    g_ = band_power(f, pxx, 30, 45)
    tot= band_power(f, pxx, 1,  50)

    fd = [
        d_, t_, a_, b_, g_, tot,
        d_/(tot+1e-10), t_/(tot+1e-10), a_/(tot+1e-10), b_/(tot+1e-10),
        a_ /(b_ +1e-10), t_ /(a_ +1e-10), (a_+b_)/(d_+t_+1e-10),
        spectral_entropy(pxx), float(f[np.argmax(pxx)]),
        float(np.sum(pxx * f) / (np.sum(pxx)+1e-10))
    ]

    return np.array(td + hj + autocorr + fd, dtype=np.float32)


# ── 2. Real-Time Processing System ─────────────────────────────

# Load the saved scaler and (optionally) your trained model
scaler = joblib.load('BCI_SCALER.joblib')
# model = joblib.load('BCI_MODEL.joblib') # Uncomment when you have your model

def process_realtime_chunk(eeg_buffer):
    """
    Takes a live buffer of raw EEG signals, extracts features, 
    and returns scaled features ready for prediction.
    """
    # 1. Extract 42 features from this single time window
    features = extract_features(np.array(eeg_buffer))
    
    # 2. Reshape to 2D array: (1 sample, 42 features)
    # This is required by Scikit-Learn since it expects a batch of inputs!
    features_2d = features.reshape(1, -1)
    
    # 3. Apply the pre-trained standard scaler
    scaled_features = scaler.transform(features_2d)
    
    # 4. Predict the intention (Uncomment once you add your model)
    # prediction = model.predict(scaled_features)
    # return prediction[0]
    
    return scaled_features

# ── 3. Simulated Live Data Stream Example ──────────────────────
if __name__ == "__main__":
    # Load raw data to act as our "live stream"
    df = pd.read_csv('eeg_precision_bci.csv')
    raw_stream = df['Signal_Raw'].values
    
    # Define window size (e.g., 128 samples = ~1 second of data)
    WINDOW_SIZE = 128
    
    print("Starting real-time simulation...\n")
    
    # Simulating processing data chunk by chunk as it arrives
    for i in range(5):
        start_idx = i * WINDOW_SIZE
        end_idx = start_idx + WINDOW_SIZE
        
        # Grab the latest chunk of data from the stream
        live_chunk = raw_stream[start_idx:end_idx]
        
        # Ensure we have enough data to process
        if len(live_chunk) == WINDOW_SIZE:
            
            # ---> THIS IS YOUR REAL-TIME PIPELINE <---
            scaled_input = process_realtime_chunk(live_chunk)
            
            print(f"Window {i+1} processed successfully.")
            print(f"Output shape: {scaled_input.shape} | Ready for model.predict()")
            # print(f"First 3 scaled features: {scaled_input[0][:3]}\n")

Starting real-time simulation...

Window 1 processed successfully.
Output shape: (1, 42) | Ready for model.predict()
Window 2 processed successfully.
Output shape: (1, 42) | Ready for model.predict()
Window 3 processed successfully.
Output shape: (1, 42) | Ready for model.predict()
Window 4 processed successfully.
Output shape: (1, 42) | Ready for model.predict()
Window 5 processed successfully.
Output shape: (1, 42) | Ready for model.predict()


In [6]:
scaled_input.shape

(1, 42)

In [7]:
import numpy as np
import pandas as pd
import joblib
import time
from scipy.signal import welch
from scipy.stats import skew, kurtosis
from scipy.integrate import trapezoid

# Set your Sampling Frequency (FS)
FS = 128 
WINDOW_SIZE = 128 # 128 samples = ~1 second of data

# ── 1. Feature Extraction Helper Functions ─────────────────────
# (Keeping these exactly the same as before)
def band_power(f, pxx, lo, hi):
    idx = (f >= lo) & (f <= hi)
    return float(trapezoid(pxx[idx], f[idx])) if idx.sum() > 0 else 0.0

def spectral_entropy(pxx):
    p = pxx / (pxx.sum() + 1e-10)
    return float(-np.sum(p * np.log2(p + 1e-10)))

def hjorth_params(x):
    d1 = np.diff(x); d2 = np.diff(d1)
    act  = float(np.var(x))
    mob  = float(np.sqrt(np.var(d1) / (act  + 1e-10)))
    comp = float(np.sqrt(np.var(d2) / (np.var(d1) + 1e-10)) / (mob + 1e-10))
    return act, mob, comp

def extract_features(ep):
    nperseg = min(128, len(ep))
    d1 = np.diff(ep)
    half = len(ep) // 2

    # Time-domain (18 features)
    td = [
        float(np.mean(ep)), float(np.std(ep)), float(np.var(ep)),
        float(skew(ep)), float(kurtosis(ep)), float(np.sqrt(np.mean(ep**2))),
        float(np.max(ep) - np.min(ep)), float(np.mean(np.abs(ep))),
        float(np.sum(np.diff(np.sign(ep)) != 0)),
        float(np.percentile(ep, 10)), float(np.percentile(ep, 25)),
        float(np.percentile(ep, 75)), float(np.percentile(ep, 90)),
        float(np.sum(np.abs(d1))),
        float(np.mean(ep[:half])), float(np.mean(ep[half:])),
        float(np.std(ep[:half])), float(np.std(ep[half:]))
    ]

    # Hjorth parameters (3 features)
    act, mob, comp = hjorth_params(ep)
    hj = [act, mob, comp]

    # Autocorrelation lags (5 features)
    ac = np.correlate(ep, ep, mode='full')
    ac = ac[len(ac)//2:]
    ac = ac / (ac[0] + 1e-10)
    autocorr = [float(ac[lag]) if lag < len(ac) else 0.0 for lag in [1, 2, 5, 10, 20]]

    # Frequency-domain (16 features)
    f, pxx = welch(ep, fs=FS, nperseg=nperseg)
    pxx = np.maximum(pxx, 1e-12)

    d_ = band_power(f, pxx, 2,  4)
    t_ = band_power(f, pxx, 4,  8)
    a_ = band_power(f, pxx, 8,  13)
    b_ = band_power(f, pxx, 13, 30)
    g_ = band_power(f, pxx, 30, 45)
    tot= band_power(f, pxx, 1,  50)

    fd = [
        d_, t_, a_, b_, g_, tot,
        d_/(tot+1e-10), t_/(tot+1e-10), a_/(tot+1e-10), b_/(tot+1e-10),
        a_ /(b_ +1e-10), t_ /(a_ +1e-10), (a_+b_)/(d_+t_+1e-10),
        spectral_entropy(pxx), float(f[np.argmax(pxx)]),
        float(np.sum(pxx * f) / (np.sum(pxx)+1e-10))
    ]

    return np.array(td + hj + autocorr + fd, dtype=np.float32)

# ── 2. Real-Time Processing System ─────────────────────────────
scaler = joblib.load('BCI_SCALER.joblib')
# model = joblib.load('BCI_MODEL.joblib') # Uncomment when you have your model

def process_realtime_chunk(eeg_buffer):
    features = extract_features(np.array(eeg_buffer))
    features_2d = features.reshape(1, -1)
    scaled_features = scaler.transform(features_2d)
    
    # prediction = model.predict(scaled_features)
    # return prediction[0]
    return scaled_features


# ── 3. Real-Time Data Stream Loop ──────────────────────────────
if __name__ == "__main__":
    df = pd.read_csv('eeg_precision_bci.csv')
    raw_stream = df['Signal_Raw'].values
    
    print("Starting continuous real-time simulation...\n")
    
    current_index = 0
    total_samples = len(raw_stream)
    window_count = 0
    
    # WHILE LOOP: Keep running as long as we have enough data for a full window
    # For actual live hardware, this would just be: `while True:`
    while current_index + WINDOW_SIZE <= total_samples:
        
        # 1. Grab the latest chunk of data (simulating a live buffer)
        live_chunk = raw_stream[current_index : current_index + WINDOW_SIZE]
        
        # 2. Process the chunk (Extract Features -> Scale -> Predict)
        scaled_input = process_realtime_chunk(live_chunk)
        
        window_count += 1
        print(f"[{window_count}] Processed window from index {current_index} to {current_index + WINDOW_SIZE}")
        
        # Optional: Print out prediction logic here
        # intent = process_realtime_chunk(live_chunk)
        # print(f"Detected Intent: {intent}")
        
        # 3. Slide the window forward
        # If you want non-overlapping windows: current_index += WINDOW_SIZE
        # If you want overlapping windows (e.g., update every 32 samples): current_index += 32
        current_index += WINDOW_SIZE 
        
        # Simulate the time it takes to gather the next second of data
        # time.sleep(1.0) 
        
        # Just to stop the console from flooding during this test, break after 10 loops
        if window_count >= 10:
            print("Stopping simulation after 10 windows...")
            break
            
    print("\nReal-time stream ended.")

Starting continuous real-time simulation...

[1] Processed window from index 0 to 128
[2] Processed window from index 128 to 256
[3] Processed window from index 256 to 384
[4] Processed window from index 384 to 512
[5] Processed window from index 512 to 640
[6] Processed window from index 640 to 768
[7] Processed window from index 768 to 896
[8] Processed window from index 896 to 1024
[9] Processed window from index 1024 to 1152
[10] Processed window from index 1152 to 1280
Stopping simulation after 10 windows...

Real-time stream ended.
